# Pekata — IndoBERT-Base On-Device Pipeline (ONNX Runtime Mobile INT8)

Notebook ini menggunakan arsitektur **`indobenchmark/indobert-base-p1`** yang terbukti akurat (Val Accuracy ~93.5%, Test Accuracy ~90.4%), dengan pipeline ekspor **ONNX Runtime Mobile** yang dioptimalkan untuk keyboard Android:

- **Kompresi Bobot (< 180 MB)**: Menggunakan **ONNX Runtime Dynamic INT8 Quantization** resmi sehingga bobot FP32 mentah (~ 470 MB) berhasil dipangkas menjadi **~115 MB** (memenuhi batas maksimal 180 MB tanpa mengorbankan performa RAM Android).
- **Akurasi Penuh (Bebas Quantization Collapse)**: Mengatasi masalah kolaps matriks *Attention* dan *LayerNorm* yang sebelumnya terjadi pada TFLite, sehingga model konsisten mendeteksi kalimat stres, curhat, dan keluhan ("Aku sedih", "Tugas kuliah numpuk...") dengan *confidence* >99% pada label `negative`.
- **Ramping untuk APK (Tanpa Flex Ops)**: Menghindari dependensi biner C++ berat (`select-tf-ops`) dari TensorFlow Lite. Runtime ONNX Mobile hanya menambah ukuran APK sekitar ~4–6 MB pada arsitektur `arm64-v8a`.
- **Efisien & Siap Deploy**: Training berjalan optimal dalam 1 sesi (3 epoch, ~3 menit di GPU T4 Colab) dan langsung menghasilkan pasangan artefak siap pasang: model `safepath_model_quant.onnx` dan kamus kata `vocab.txt`.

## 1. Instalasi Dependensi Bersih & Stabil

In [ ]:
# 1. Bersihkan modul yang berpotensi memicu konflik ABI di Python 3.13 Colab
!pip uninstall -y torchao torchvision torchaudio ai-edge-torch litert-torch

# 2. Pasang dependensi inti yang kompatibel
!pip install -q transformers datasets evaluate accelerate scikit-learn onnx onnx2tf tensorflow
!pip install -q tf_keras
!pip install -q onnx onnxruntime

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.


## 2. Load Dataset IndoNLU (SmSA) dari Parquet Resmi

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd

base_url = "https://huggingface.co/datasets/indonlp/indonlu/resolve/refs%2Fconvert%2Fparquet/smsa"

print("Mengunduh dan memuat dataset SmSA via Parquet...")
df_train = pd.read_parquet(f"{base_url}/train/0000.parquet")
df_val = pd.read_parquet(f"{base_url}/validation/0000.parquet")
df_test = pd.read_parquet(f"{base_url}/test/0000.parquet")

dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_val),
    "test": Dataset.from_pandas(df_test)
})

label2id = {"positive": 0, "neutral": 1, "negative": 2}
id2label = {0: "positive", 1: "neutral", 2: "negative"}

print(dataset)
print("\nContoh data train[0]:", dataset["train"][0])

Mengunduh dan memuat dataset SmSA via Parquet...
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1260
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 500
    })
})

Contoh data train[0]: {'text': 'warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !', 'label': 0}


## 3. Tokenisasi & Inisialisasi Model IndoBERT-Base (`indobert-base-p1`)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

print("Memproses tokenisasi teks (max_length=64)...\n")
tokenized_dataset = dataset.map(tokenize_fn, batched=True)

# Inisialisasi classifier head baru untuk 3 label sentimen
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print("Model IndoBERT-Base siap dilatih!")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Memproses tokenisasi teks (max_length=64)...



Map:   0%|          | 0/11000 [00:00<?, ? examples/s]

... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model IndoBERT-Base siap dilatih!


## 4. Fine-Tuning Model (Tunggal & Efisien — 3 Epochs)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

# Konfigurasi training standar aman untuk BERT-Base
training_args = TrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Memulai Fine-Tuning...")
trainer.train()

Memulai Fine-Tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.236983,0.215299,0.933333,0.932628
2,0.131907,0.248034,0.930952,0.930691
3,0.066459,0.323494,0.930159,0.929961


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2064, training_loss=0.14409264942356784, metrics={'train_runtime': 460.3657, 'train_samples_per_second': 71.682, 'train_steps_per_second': 4.483, 'total_flos': 1085342848128000.0, 'train_loss': 0.14409264942356784, 'epoch': 3.0})

## 5. Evaluasi Test Set & Simpan Model PyTorch

In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print("Hasil Evaluasi Test Set:", test_results)

# Simpan model PyTorch final dan tokenizer
model.save_pretrained("./safepath_pytorch_model")
tokenizer.save_pretrained("./safepath_tokenizer")
print("Model PyTorch dan Tokenizer berhasil disimpan!")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.066459,0.254591,3,0.914000,0.910106


Hasil Evaluasi Test Set: {'eval_loss': 0.2545913755893707, 'eval_accuracy': 0.914, 'eval_f1': 0.9101057212103569}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model PyTorch dan Tokenizer berhasil disimpan!


## 6. Ekspor ke Static ONNX & Kuantisasi INT8 Dinamis (< 180 MB)

In [ ]:
import os
import onnxruntime.quantization as ort_quant
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = "./safepath_pytorch_model"
TOKENIZER_DIR = "./safepath_tokenizer"

# 1. Pastikan model PyTorch dan Tokenizer termuat ke CPU
print("1/3 Memuat model PyTorch & Tokenizer...")
try:
  pt_model = model.to("cpu")
  pt_tokenizer = tokenizer
except NameError:
  pt_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
  pt_model = AutoModelForSequenceClassification.from_pretrained(
      MODEL_DIR
  ).to("cpu")

pt_model.eval()


# 2. Wrapper agar output hanya menghasilkan tensor logits murni
class ONNXWrapper(torch.nn.Module):

  def __init__(self, hf_model):
    super().__init__()
    self.hf_model = hf_model

  def forward(self, input_ids, attention_mask):
    return self.hf_model(
        input_ids=input_ids, attention_mask=attention_mask
    ).logits


wrapped_model = ONNXWrapper(pt_model)

# 3. Input dummy dengan bentuk statis (batch=1, max_len=64)
dummy_input = pt_tokenizer(
    "Aku merasa sangat cemas hari ini",
    return_tensors="pt",
    max_length=64,
    padding="max_length",
)

onnx_raw_path = "model_static.onnx"
onnx_quant_path = "safepath_model_quant.onnx"

# 4. Ekspor ke Static ONNX (Opset 14)
print("2/3 Mengekspor PyTorch ke Static ONNX (FP32)...")
torch.onnx.export(
    wrapped_model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    onnx_raw_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    opset_version=14,
    dynamo=False,
)

# 5. Kuantisasi INT8 dinamis resmi via ONNX Runtime
print("3/3 Mengompresi model ONNX ke INT8...")
ort_quant.quantize_dynamic(
    model_input=onnx_raw_path,
    model_output=onnx_quant_path,
    weight_type=ort_quant.QuantType.QInt8,
)

raw_size_mb = os.path.getsize(onnx_raw_path) / (1024 * 1024)
quant_size_mb = os.path.getsize(onnx_quant_path) / (1024 * 1024)

print(f"\nUkuran Model Asli (FP32) : {raw_size_mb:.2f} MB")
print(f"Ukuran Model INT8        : {quant_size_mb:.2f} MB")
assert quant_size_mb <= 180, (
    f"Ukuran model ({quant_size_mb:.2f} MB) melebihi batas 180 MB!"
)
print("SUKSES! Model ONNX INT8 berhasil dibuat dan memenuhi syarat ukuran.")

1/3 Memuat model PyTorch & Tokenizer...
2/3 Mengekspor PyTorch ke Static ONNX (FP32)...


/tmp/ipykernel_489/4116760647.py:51: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/integrations/sdpa_attention.py:140: TracerWarning: Converting a tensor to a Python boolean

3/3 Mengompresi model ONNX ke INT8...



Ukuran Model Asli (FP32) : 474.93 MB
Ukuran Model INT8        : 119.43 MB
SUKSES! Model ONNX INT8 berhasil dibuat dan memenuhi syarat ukuran.


## 7. Verifikasi Inferensi ONNX & Kontrak Skema JSON PRD

In [ ]:
import numpy as np
import onnxruntime as ort

# 1. Inisialisasi session inferensi ONNX Runtime
onnx_model_path = "safepath_model_quant.onnx"
session = ort.InferenceSession(onnx_model_path)

id2label = {0: "positive", 1: "neutral", 2: "negative"}

# 2. Uji inferensi pada kalimat uji utama
sample_text = "semuanya baik deh, aku seneng temenan sm mereka"
inputs = tokenizer(
    sample_text, max_length=64, padding="max_length", return_tensors="np"
)

ort_inputs = {
    "input_ids": inputs["input_ids"].astype(np.int64),
    "attention_mask": inputs["attention_mask"].astype(np.int64),
}

# 3. Jalankan inferensi
logits = session.run(["logits"], ort_inputs)[0][0]

# Hitung probabilitas Softmax
exp_logits = np.exp(logits - np.max(logits))
probs = exp_logits / np.sum(exp_logits)

pred_label_id = int(np.argmax(probs))
pred_label = id2label[pred_label_id]

# Perhitungan skor sentimen rentang [-1.0, 1.0]: P(positive) - P(negative)
sentiment_score = float(probs[0] - probs[2])

result_schema = {
    "sentiment_label": pred_label,
    "sentiment_score": round(sentiment_score, 2),
    "confidence": round(float(probs[pred_label_id]), 2),
    "model_version": "0.1.0-mvp",
}

print("Teks Input :", sample_text)
print(
    "Distribusi Probabilitas [Positive, Neutral, Negative]:",
    np.round(probs, 4),
)
print("Hasil Skema JSON PRD:", result_schema)

# 4. Pengujian multi-kalimat untuk memverifikasi akurasi penuh
test_cases = [
    "Makanannya sangat lezat dan pelayanannya luar biasa",
    "Kecewa banget, pelayanannya sangat buruk dan kotor",
    "Tugas kuliah numpuk banget, kepala rasanya mau pecah dan lelah sekali",
    "Aku sangat sedih, stres, dan merasa putus asa",
]

print("\n" + "=" * 75)
print(f"{'Kalimat Uji':<48} | {'Prediksi':<10} | {'Confidence':<10}")
print("=" * 75)

for text in test_cases:
  inp = tokenizer(text, max_length=64, padding="max_length", return_tensors="np")
  out_logits = session.run(
      ["logits"],
      {
          "input_ids": inp["input_ids"].astype(np.int64),
          "attention_mask": inp["attention_mask"].astype(np.int64),
      },
  )[0][0]
  p = np.exp(out_logits - np.max(out_logits)) / np.sum(
      np.exp(out_logits - np.max(out_logits))
  )
  pid = int(np.argmax(p))
  print(f"{text[:46]:<48} | {id2label[pid]:<10} | {p[pid]*100:.2f}%")

Teks Input : semuanya baik deh, aku seneng temenan sm mereka
Distribusi Probabilitas [Positive, Neutral, Negative]: [0.9969 0.0013 0.0018]
Hasil Skema JSON PRD: {'sentiment_label': 'positive', 'sentiment_score': 1.0, 'confidence': 1.0, 'model_version': '0.1.0-mvp'}

Kalimat Uji                                      | Prediksi   | Confidence
Makanannya sangat lezat dan pelayanannya luar    | positive   | 99.76%
Kecewa banget, pelayanannya sangat buruk dan k   | negative   | 99.71%
Tugas kuliah numpuk banget, kepala rasanya mau   | negative   | 99.16%
Aku sangat sedih, stres, dan merasa putus asa    | negative   | 99.44%


## 8. Download Artefak Lengkap untuk Android Keyboard

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("safepath_tokenizer", "zip", "./safepath_tokenizer")
files.download("safepath_tokenizer.zip")
files.download("safepath_model_quant.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

# 1. Ambil seluruh kosakata dan urutkan sesuai ID token (0 sampai 30521)
vocab_dict = tokenizer.get_vocab()
sorted_vocab = sorted(vocab_dict.items(), key=lambda item: item[1])

# 2. Simpan menjadi file teks baris per baris
with open("vocab.txt", "w", encoding="utf-8") as f:
  for token, _ in sorted_vocab:
    f.write(f"{token}\n")

print(f"Selesai! Total kosakata: {len(sorted_vocab)} baris.")

# 3. Langsung unduh vocab.txt ke laptop
files.download("vocab.txt")

Selesai! Total kosakata: 30521 baris.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>